# Warm-up: CPU vs GPU Benchmark

**AI in Medicine Workshop**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/SCAI-4EUWorkshopAIinMedicineWorkshop/blob/main/Hands-On-Session-warmup/cpu_gpu_benchmark.ipynb)

This warm-up measures how much faster a GPU is than a CPU on two workloads:
a large matrix multiplication and a BERT forward pass.

> **Enable the GPU first:** in Colab go to **Runtime > Change runtime type > Hardware accelerator > GPU**, then run the cells. Without a GPU the benchmark still runs but only reports CPU timings.

## 1. Matrix Multiplication

Multiply two `4096 x 4096` matrices 10 times on the CPU, then (if available) on the GPU, and compare the average time per multiplication.

In [ ]:
import torch
import time

# Matrix size
n = 4096

# Create random matrices
a_cpu = torch.randn(n, n)
b_cpu = torch.randn(n, n)

# CPU benchmark
start = time.time()
for _ in range(10):
    c_cpu = torch.matmul(a_cpu, b_cpu)
cpu_time = (time.time() - start) / 10
print(f"CPU: {cpu_time*1000:.2f} ms per multiplication")

# Check if CUDA is available
if torch.cuda.is_available():
    # Move to GPU
    a_gpu = a_cpu.to('cuda')
    b_gpu = b_cpu.to('cuda')

    # Warmup
    for _ in range(3):
        _ = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()

    # GPU benchmark
    start = time.time()
    for _ in range(10):
        c_gpu = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()
    gpu_time = (time.time() - start) / 10

    print(f"GPU: {gpu_time*1000:.2f} ms per multiplication")
    print(f"Speedup: {cpu_time/gpu_time:.1f}x")
else:
    print("CUDA not available. Enable GPU in Colab: Runtime > Change runtime type > GPU")

## 2. BERT Forward Pass

Run a batch of 32 sentences through `bert-base-uncased` on the CPU vs the GPU.
This is closer to a real deep-learning workload than a raw matmul.

In [ ]:
# transformers ships with Colab, but install it explicitly to be safe
!pip install -q transformers

In [ ]:
import torch
import time
from transformers import AutoTokenizer, AutoModel

# Load model and tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Sample text
texts = ["This is a test sentence for benchmarking."] * 32  # batch of 32

# Tokenize
inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)

# CPU benchmark
model_cpu = model.to("cpu")
inputs_cpu = {k: v.to("cpu") for k, v in inputs.items()}

with torch.no_grad():
    start = time.time()
    for _ in range(10):
        _ = model_cpu(**inputs_cpu)
    cpu_time = (time.time() - start) / 10
print(f"CPU: {cpu_time*1000:.2f} ms per forward pass")

# GPU benchmark
if torch.cuda.is_available():
    model_gpu = model.to("cuda")
    inputs_gpu = {k: v.to("cuda") for k, v in inputs.items()}

    # Warmup
    with torch.no_grad():
        for _ in range(3):
            _ = model_gpu(**inputs_gpu)
        torch.cuda.synchronize()

    # Benchmark
    with torch.no_grad():
        start = time.time()
        for _ in range(10):
            _ = model_gpu(**inputs_gpu)
        torch.cuda.synchronize()
        gpu_time = (time.time() - start) / 10

    print(f"GPU: {gpu_time*1000:.2f} ms per forward pass")
    print(f"Speedup: {cpu_time/gpu_time:.1f}x")
else:
    print("CUDA not available. Enable GPU in Colab: Runtime > Change runtime type > GPU")